# unnoise on Google Colab

**GitHub does not add an “Open in Colab” button to notebook files.** Use the badge in the repo README, paste this link in your browser, or in Colab: *File → Open notebook → GitHub* → `sivaratrisrinivas/unnoise` → `colab/unnoise.ipynb`.

https://colab.research.google.com/github/sivaratrisrinivas/unnoise/blob/main/colab/unnoise.ipynb

---

1. **Runtime → Change runtime type → GPU**
2. Run the cells in order.
3. The last cell waits until port **8000** is open, then embeds the UI. If you ever see a **blank iframe**, the old `sleep(10)` pattern was too short while the model downloaded (`WARMUP_MODEL=1`); this notebook uses `WARMUP_MODEL=0` so the server binds fast and the first **Generate** loads weights.

If `git clone` fails (private repo), zip this project on your machine, **Upload** it in Colab, unzip, and `%cd` into the folder instead of cloning.

In [ ]:
!pip -q install -U pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install diffusers transformers safetensors pillow

In [ ]:
# Replace with your repo if different; or upload a zip and skip this cell.
!git clone https://github.com/sivaratrisrinivas/unnoise.git
%cd unnoise

In [ ]:
import os
import socket
import subprocess
import sys
import time

from google.colab import output

# Must match where `git clone` put the repo (previous cell uses %cd unnoise).
REPO = "/content/unnoise"
assert os.path.isfile(os.path.join(REPO, "app.py")), (
    f"Expected {REPO}/app.py — run the clone + %cd cell first, or set REPO to your folder."
)

LOG_PATH = "/tmp/unnoise_app.log"
env = os.environ.copy()
env["HOST"] = "0.0.0.0"
env["PORT"] = "8000"
env["DEVICE"] = "cuda"
env["UI_MAX_FRAMES"] = "all"
# WARMUP_MODEL=1 loads HF weights *before* the HTTP server binds — first run can take
# many minutes and a fixed sleep(10) leaves nothing on :8000 → blank iframe.
# 0 = server starts immediately; first "Generate" pays the download cost.
env["WARMUP_MODEL"] = "0"

with open(LOG_PATH, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        [sys.executable, "app.py"],
        cwd=REPO,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )


def wait_for_port(port: int, timeout_s: float = 120) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1.0):
                return True
        except OSError:
            time.sleep(1)
            print(".", end="", flush=True)
    return False


print("Starting app.py — logging to", LOG_PATH)
print("Waiting for port 8000 (max ~2 min) …", end="", flush=True)
if not wait_for_port(8000, timeout_s=120):
    print("\n--- server log (tail) ---")
    print(open(LOG_PATH, "r", encoding="utf-8", errors="replace").read()[-8000:])
    raise RuntimeError("Nothing listening on 8000 — see log above.")

print("\nServer is up. Embedding UI…")
output.serve_kernel_port_as_iframe(8000, path="/", height=900)